# Exploratory Data Analysis — Inside Airbnb London

**City:** London | **Snapshot:** 2025-09-14 | **Currency:** GBP  
**Data source:** `data/processed/london/` (cleaned + enriched Parquet files)

## Business questions this notebook answers

1. How are prices distributed, and which boroughs / room types command a premium?
2. Where are listings concentrated geographically?
3. How do hosts segment (casual vs professional) and how does it affect price and quality?
4. What does review activity signal about demand?
5. Is there seasonal or weekday/weekend variation in availability?
6. Are there unusual listings (very long stays, very high price, inactive)?
7. What is the relationship between review score, response rate, and occupancy proxy?

> **A-005 constraint:** `calendar.price` and `calendar.adjusted_price` are 100 % NULL in this snapshot.  
> All temporal analyses use **availability** and **occupancy proxy** instead of price.

---
## Step 1 — Load and verify the data

In [1]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ── Paths ────────────────────────────────────────────────────────────────────
PROCESSED = Path("../data/processed/london")
RAW       = Path("../data/raw/london")
FIGS      = Path("../reports/figures/eda")
TABLES    = Path("../reports/tables")

# Output directories (created here so later cells never fail on missing dir)
FIGS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# ── Plot style ───────────────────────────────────────────────────────────────
# Consistent, clean look for all charts
sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

SNAPSHOT_DATE = pd.Timestamp("2025-09-14")
CITY_LABEL    = "London"

In [2]:
# ── Load datasets ─────────────────────────────────────────────────────────────
# listing_master = one row per listing, all derived metrics already joined
listings = pd.read_parquet(PROCESSED / "listing_master.parquet")

# calendar = one row per (listing, date) — 35M rows; load only needed columns
# to avoid exhausting RAM. calendar_clean.parquet has no price column (A-005).
calendar = pd.read_parquet(
    PROCESSED / "calendar_clean.parquet",
    columns=["listing_id", "date", "available", "minimum_nights", "maximum_nights"],
)
# Derive numeric flag: 1 = available, 0 = blocked/booked
calendar["available_int"] = calendar["available"].astype(int)

# reviews = one row per review, 2.1M rows
reviews = pd.read_parquet(PROCESSED / "reviews_clean.parquet")

# neighbourhood_summary = 33 boroughs, pre-aggregated metrics
neigh_sum = pd.read_parquet(PROCESSED / "neighbourhood_summary.parquet")

print(f"listings  : {listings.shape[0]:>9,} rows × {listings.shape[1]} cols")
print(f"calendar  : {calendar.shape[0]:>9,} rows × {calendar.shape[1]} cols")
print(f"reviews   : {reviews.shape[0]:>9,} rows × {reviews.shape[1]} cols")
print(f"neigh_sum : {neigh_sum.shape[0]:>9,} rows × {neigh_sum.shape[1]} cols")

listings  :    96,871 rows × 100 cols
calendar  : 35,357,974 rows × 6 cols
reviews   : 2,097,996 rows × 8 cols
neigh_sum :        33 rows × 12 cols


In [3]:
# ── Uniqueness and null guards ────────────────────────────────────────────────
# These must hold before any analysis. If they raise, stop and investigate.
assert listings["id"].is_unique,       "listing id not unique — data issue"
assert not listings["id"].isna().any(), "listing id has nulls — data issue"

print("Uniqueness checks passed")

Uniqueness checks passed


In [4]:
# ── Key column overview ───────────────────────────────────────────────────────
# Quick look at the numeric columns that drive the rest of the EDA
key_cols = [
    "price_numeric",        # nightly price in GBP (NULL for 36% of listings)
    "accommodates",         # guest capacity
    "bedrooms",             # bedroom count
    "bathroom_count",       # parsed from bathrooms_text
    "minimum_nights",       # host-set minimum stay
    "availability_365",     # days open per year (from listings snapshot)
    "number_of_reviews",    # total reviews ever
    "reviews_per_month_calc",  # computed rate over active months
    "review_scores_rating", # overall rating (0–5 scale in this data)
    "host_tenure_years",    # years since host joined
    "host_portfolio_size",  # number of listings this host has
    "occupancy_proxy",      # 1 − availability_rate (calendar-based)
    "revenue_proxy_gbp",    # price_numeric × unavailable_days (upper bound)
]

listings[key_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T.round(2)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
price_numeric,61963.0,229.92,4437.59,7.00,30.00,40.00,77.00,135.00,221.00,500.00,1100.00,1085147.00
accommodates,96871.0,3.33,2.08,1.00,1.00,1.00,2.00,2.00,4.00,7.00,10.00,16.00
bedrooms,84096.0,1.61,1.04,0.00,0.00,1.00,1.00,1.00,2.00,4.00,5.00,50.00
bathroom_count,96718.0,1.33,0.68,0.00,0.50,1.00,1.00,1.00,1.50,2.50,3.50,30.00
minimum_nights,96871.0,5.44,23.69,1.00,1.00,1.00,1.00,2.00,4.00,14.00,90.00,1125.00
availability_365,96871.0,144.93,141.81,0.00,0.00,0.00,0.00,96.00,288.00,364.00,365.00,365.00
number_of_reviews,96871.0,21.66,50.37,0.00,0.00,0.00,1.00,5.00,20.00,98.00,246.00,1902.00
reviews_per_month_calc,72749.0,0.64,1.11,0.00,0.00,0.00,0.00,0.17,0.75,2.92,5.33,32.50
review_scores_rating,72749.0,4.68,0.49,0.00,2.50,4.00,4.58,4.83,5.00,5.00,5.00,5.00
host_tenure_years,96830.0,7.64,3.89,0.01,0.27,0.91,4.06,8.57,10.58,13.20,14.30,17.05


In [5]:
# ── Missingness summary for key columns ──────────────────────────────────────
miss = (
    listings[key_cols]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)
miss["missing_pct"] = (miss["missing_count"] / len(listings) * 100).round(1)

print(miss.to_string())

                        missing_count  missing_pct
price_numeric                   34908         36.0
accommodates                        0          0.0
bedrooms                        12775         13.2
bathroom_count                    153          0.2
minimum_nights                      0          0.0
availability_365                    0          0.0
number_of_reviews                   0          0.0
reviews_per_month_calc          24122         24.9
review_scores_rating            24122         24.9
host_tenure_years                  41          0.0
host_portfolio_size                 0          0.0
occupancy_proxy                     0          0.0
revenue_proxy_gbp               34908         36.0

In [6]:
# ── Categorical field overview ────────────────────────────────────────────────
cat_cols = ["room_type", "property_type_bucket", "neighbourhood_cleansed"]
for col in cat_cols:
    print(f"\n{col} ({listings[col].nunique()} unique):")
    print(listings[col].value_counts().head(10).to_string())


room_type (4 unique):
room_type
entire_home     62907
private_room    33643
shared_room       212
hotel_room        109

property_type_bucket (5 unique):
property_type_bucket
apartment    69713
house        24244
hotel         2107
other          615
unique         192

neighbourhood_cleansed (33 unique):
neighbourhood_cleansed
Westminster               11385
Tower Hamlets              7469
Camden                     6551
Kensington and Chelsea     6401
Hackney                    6359
Southwark                  5475
Lambeth                    5190
Islington                  5036
Wandsworth                 4965
Hammersmith and Fulham     4157


In [7]:
# ── Calendar data sanity ──────────────────────────────────────────────────────
# Confirm A-005: calendar price is absent from this parquet (only 4 cols loaded)
# available_int = 1 means available, 0 means blocked/booked
print("Calendar date range:",
      calendar["date"].min(), "→", calendar["date"].max())
print("available_int values:",
      calendar["available_int"].value_counts().to_dict())
print("Unique listing IDs in calendar:",
      calendar["listing_id"].nunique())

Calendar date range: 2025-09-14 00:00:00 → 2026-09-17 00:00:00


available_int values: {0: 21313866, 1: 14044108}


Unique listing IDs in calendar: 96871


In [8]:
# ── Reviews data sanity ───────────────────────────────────────────────────────
print("Reviews date range:",
      reviews["date"].min(), "→", reviews["date"].max())
print("Unique listings with reviews:",
      reviews["listing_id"].nunique())
print("Reviews columns:", list(reviews.columns))

Reviews date range: 2009-12-21 00:00:00 → 2025-09-17 00:00:00
Unique listings with reviews: 72749
Reviews columns: ['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments', 'comment_length', 'comment_is_duplicate']


In [9]:
# ── Save full numerical summary to CSV ───────────────────────────────────────
# This CSV backs the 'Summary Statistics' section of eda_findings.md
summary = listings[key_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T.round(4)
summary["missing_count"] = listings[key_cols].isna().sum()
summary["missing_pct"]   = (listings[key_cols].isna().mean() * 100).round(1)
summary.to_csv(TABLES / "numerical_summary.csv")

print("Saved → reports/tables/numerical_summary.csv")

Saved → reports/tables/numerical_summary.csv


In [10]:
# ── Analysis population: price-eligible listings ──────────────────────────────
# Keep only listings with a valid, non-negative price.
# This subset is used for all price-related charts.
# Non-price analyses (availability, reviews) use the full `listings` DataFrame.
eda = listings[
    listings["price_numeric"].notna()
    & listings["price_numeric"].ge(0)
].copy()

# Document the exclusions clearly
excl = pd.DataFrame([
    {"rule": "total_listings",         "count": len(listings)},
    {"rule": "missing_price",          "count": int(listings["price_numeric"].isna().sum())},
    {"rule": "negative_price",         "count": int((listings["price_numeric"] < 0).sum())},
    {"rule": "price_eligible_population", "count": len(eda)},
])
excl.to_csv(TABLES / "population_exclusions.csv", index=False)

print(excl.to_string(index=False))
print(f"\nPrice-eligible coverage: {len(eda)/len(listings)*100:.1f}% of all listings")

                     rule  count
           total_listings  96871
            missing_price  34908
           negative_price      0
price_eligible_population  61963

Price-eligible coverage: 64.0% of all listings
